In [ ]:
# =============================================================================
# MycoIris — Pairwise & Batch Hamming Distance Analysis
# =============================================================================
#
# Computes mask-aware Hamming distances between mycocode binary iris codes
# stored as *_mycocode_bits.npz files produced by the MycoIris encoding pipeline.
#
# Two operating modes:
#   1. Single-pair  — compare two specific NPZ files with per-channel,
#                     per-filter, and rotation-sweep diagnostics.
#   2. Batch        — compute all pairwise distances for every NPZ file
#                     in a given folder.
#
# All file paths in the USER INPUTS section below should be updated to match
# the location of your data before running.
# =============================================================================

import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

# ----- USER INPUTS -----------------------------------------------------------
#
# Single-pair mode: set both paths to compare two specific samples.
# Leave as None to skip single-pair mode.
NPZ_PATH_1 = None
NPZ_PATH_2 = None

# Batch mode: set to the folder containing *_mycocode_bits.npz files.
# Set to None to skip batch mode.
MYCOCODE_FOLDER = "./data"

# Maximum angular column shift to test for rotation robustness.
MAX_SHIFT_COLS = 200

# Plot toggles (single-pair mode only)
PLOT_PER_CHANNEL_BAR  = True
PLOT_ROTATION_SWEEP   = True
PLOT_MISMATCH_HEATMAP = True

# Batch mode: histogram of hamming_min across all pairs
PLOT_BATCH_HIST = True
# -----------------------------------------------------------------------------


# =============================================================================
# Utility Functions
# =============================================================================

def load_mycocode(npz_path: str):
    """Load a standardized mycocode_bits.npz and return code, mask, metadata."""
    if not isinstance(npz_path, (str, bytes, os.PathLike)):
        raise TypeError(f"Expected a path-like for npz_path, got {type(npz_path)}")
    data = np.load(npz_path, allow_pickle=True)
    code = data["code_uint2"]
    mask = data["mask"]
    meta = {
        "filters": data["filters"],
        "channels": data["channels"],
        "channel_set": data["channel_set"],
        "R": int(data["R"]), "A": int(data["A"])
    }
    return code, mask, meta

def assert_compatible(meta1, meta2):
    """Verify that two mycocode volumes share the same grid and filter bank."""
    assert meta1["R"] == meta2["R"] and meta1["A"] == meta2["A"], \
        f"(R,A) mismatch: {meta1['R']},{meta1['A']} vs {meta2['R']},{meta2['A']}"
    f1, f2 = meta1["filters"], meta2["filters"]
    c1 = np.array([str(x) for x in meta1["channels"]])
    c2 = np.array([str(x) for x in meta2["channels"]])
    assert f1.shape == f2.shape and np.allclose(f1, f2), "Filter bank mismatch."
    assert c1.shape == c2.shape and np.all(c1 == c2), "Channels ordering mismatch."

def hamming_maskaware(code1, code2, mask1, mask2):
    """
    Compute mask-aware Hamming distance between two code volumes.

    Parameters
    ----------
    code1, code2 : ndarray, shape (R, A, Nf), uint8 in [0..3]
    mask1, mask2 : ndarray, shape (R, A, Nf), binary validity masks

    Returns
    -------
    hamming    : float — fraction of differing valid bits
    valid_mask : bool array — positions where both codes are valid
    diff_mask  : bool array — positions that differ and are valid
    """
    valid = (mask1.astype(bool) & mask2.astype(bool))
    diff = (code1 != code2) & valid
    total_valid = np.count_nonzero(valid)
    total_diff  = np.count_nonzero(diff)
    hamming = (total_diff / total_valid) if total_valid > 0 else np.nan
    return hamming, valid, diff

def per_filter_hamming(code1, code2, mask1, mask2):
    """Return per-filter mask-aware Hamming distances, shape (Nf,)."""
    Nf = code1.shape[2]
    out = np.full(Nf, np.nan, dtype=np.float32)
    for f in range(Nf):
        h, _, _ = hamming_maskaware(code1[:, :, f], code2[:, :, f],
                                    mask1[:, :, f], mask2[:, :, f])
        out[f] = h
    return out

def per_channel_hamming(per_filter_vals, channels):
    """
    Group per-filter Hamming values by channel name.

    Returns a dict mapping each channel to its mean Hamming distance.
    """
    channels = np.array([str(x) for x in channels])
    uniq = np.unique(channels)
    result = {}
    for ch in uniq:
        vals = per_filter_vals[channels == ch]
        result[ch] = np.nanmean(vals) if np.any(~np.isnan(vals)) else np.nan
    return result

def sweep_rotations(code1, code2, mask1, mask2, max_shift=8):
    """
    Circularly shift code2/mask2 across the angular axis to find the
    rotation offset that minimizes Hamming distance.

    Returns
    -------
    best_shift : int
    best_score : float
    shifts     : ndarray of tested shift values
    scores     : ndarray of corresponding Hamming distances
    """
    A = code1.shape[1]
    shifts = np.arange(-max_shift, max_shift + 1, dtype=int)
    scores = np.full_like(shifts, fill_value=np.nan, dtype=float)
    for i, k in enumerate(shifts):
        c2s = np.roll(code2, shift=k, axis=1)
        m2s = np.roll(mask2, shift=k, axis=1)
        h, _, _ = hamming_maskaware(code1, c2s, mask1, m2s)
        scores[i] = h
    best_idx = int(np.nanargmin(scores))
    return int(shifts[best_idx]), float(scores[best_idx]), shifts, scores

def degrees_per_col(A):
    """Convert angular column index to degrees."""
    return 360.0 / float(A) if A else np.nan


# =============================================================================
# Single-Pair Comparison
# =============================================================================

if NPZ_PATH_1 and NPZ_PATH_2:
    code1, mask1, meta1 = load_mycocode(NPZ_PATH_1)
    code2, mask2, meta2 = load_mycocode(NPZ_PATH_2)

    assert_compatible(meta1, meta2)
    R, A, Nf = code1.shape
    channels = np.array([str(x) for x in meta1["channels"]])

    # Global (unshifted) Hamming distance
    h_global, valid_mask, diff_mask = hamming_maskaware(code1, code2, mask1, mask2)
    valid_coverage = np.count_nonzero(valid_mask) / valid_mask.size
    print(f"Global Hamming (no shift): {h_global:.6f} | Valid coverage: {valid_coverage:.3f}")

    # Per-filter and per-channel breakdown
    pf = per_filter_hamming(code1, code2, mask1, mask2)
    pc = per_channel_hamming(pf, channels)

    df_filters = pd.DataFrame({
        "filter_index": np.arange(Nf),
        "channel": channels,
        "sigma": meta1["filters"][:, 0],
        "theta_rad": meta1["filters"][:, 1],
        "theta_deg": np.degrees(meta1["filters"][:, 1]),
        "hamming": pf
    })
    print("\nPer-filter Hamming (first 10 rows):")
    display(df_filters.head(10))

    df_channels = pd.DataFrame({"channel": list(pc.keys()), "hamming": list(pc.values())})
    print("\nPer-channel Hamming:")
    display(df_channels)

    # Rotation sweep
    best_shift_cols, h_min, shifts, scores = sweep_rotations(code1, code2, mask1, mask2, MAX_SHIFT_COLS)
    deg_col = degrees_per_col(A)
    print(f"\nRotation sweep: best shift = {best_shift_cols:+d} cols "
          f"({best_shift_cols * deg_col:+.2f}\u00b0), Hamming_min = {h_min:.6f}")

    # --- Plots ---
    if PLOT_PER_CHANNEL_BAR:
        plt.figure(figsize=(6, 3.5))
        plt.bar(df_channels["channel"], df_channels["hamming"])
        plt.ylabel("Hamming Distance")
        plt.title("Per-channel Hamming Distance (unshifted)")
        plt.ylim(0, 1)
        plt.tight_layout()
        plt.show()

    if PLOT_ROTATION_SWEEP:
        plt.figure(figsize=(6, 3.5))
        plt.plot(shifts, scores, marker="o")
        plt.axvline(best_shift_cols, linestyle="--")
        plt.xlabel("Angular shift (columns)")
        plt.ylabel("Hamming Distance")
        plt.title("Rotation Robustness Sweep")
        plt.tight_layout()
        plt.show()

    if PLOT_MISMATCH_HEATMAP:
        mismatch_fraction = diff_mask.mean(axis=2).astype(np.float32)
        plt.figure(figsize=(9, 3))
        im = plt.imshow(mismatch_fraction, cmap="magma", aspect="auto", vmin=0, vmax=1)
        plt.title("Average bit mismatch per pixel (across filters) — unshifted")
        plt.axis("off")
        plt.colorbar(im, fraction=0.02, pad=0.02, label="Mismatch fraction")
        plt.tight_layout()
        plt.show()
else:
    print("Single-pair comparison skipped (provide NPZ_PATH_1 and NPZ_PATH_2 to enable).")


# =============================================================================
# Batch Pairwise Comparison
# =============================================================================

if MYCOCODE_FOLDER and os.path.isdir(MYCOCODE_FOLDER):
    files = [f for f in os.listdir(MYCOCODE_FOLDER) if f.endswith("_mycocode_bits.npz")]
    files = sorted(files)
    print(f"\nFound {len(files)} NPZ file(s) in: {MYCOCODE_FOLDER}")
    pairs = [(f1, f2) for i, f1 in enumerate(files) for f2 in files[i:]]
    rows = []

    if len(files) < 2:
        print("Need >= 2 '*_mycocode_bits.npz' files for batch mode.")
    else:
        for f1, f2 in pairs:
            p1 = os.path.join(MYCOCODE_FOLDER, f1)
            p2 = os.path.join(MYCOCODE_FOLDER, f2)
            try:
                c1, m1, mt1 = load_mycocode(p1)
                c2, m2, mt2 = load_mycocode(p2)
                assert_compatible(mt1, mt2)
                h0, V, _ = hamming_maskaware(c1, c2, m1, m2)
                best_k, hmin, shs, scs = sweep_rotations(c1, c2, m1, m2, MAX_SHIFT_COLS)
                rows.append({
                    "sample1": f1, "sample2": f2,
                    "hamming_0": h0,
                    "hamming_min": hmin,
                    "best_shift_cols": best_k,
                    "best_shift_deg": best_k * degrees_per_col(mt1["A"]),
                    "valid_coverage": np.count_nonzero(V) / V.size
                })
            except Exception as e:
                rows.append({
                    "sample1": f1, "sample2": f2,
                    "error": str(e)
                })

        df_pairs = pd.DataFrame(rows)
        print("\nBatch pairwise summary (first 10 rows):")
        display(df_pairs.head(10))

        if "hamming_min" in df_pairs.columns:
            df_ok = df_pairs.dropna(subset=["hamming_min"]).copy()
            if not df_ok.empty:
                print("\nTop-10 closest pairs (lowest hamming_min):")
                display(df_ok.sort_values("hamming_min").head(10))
                print("\nTop-10 farthest pairs (highest hamming_min):")
                display(df_ok.sort_values("hamming_min", ascending=False).head(10))

        if PLOT_BATCH_HIST and "hamming_min" in df_pairs.columns:
            vals = df_pairs["hamming_min"].dropna().values
            if vals.size:
                plt.figure(figsize=(6, 3.5))
                plt.hist(vals, bins=20, edgecolor="k")
                plt.xlabel("hamming_min")
                plt.ylabel("Count")
                plt.title("Distribution of rotation-robust Hamming distances")
                plt.tight_layout()
                plt.show()
else:
    print(f"Batch mode skipped or folder not found: {MYCOCODE_FOLDER}")


In [ ]:
# =============================================================================
# Pairwise Hamming Distance Heatmap
# =============================================================================
#
# Builds a symmetric NxN distance matrix from the batch results (df_pairs)
# and renders it as a heatmap. Saves the figure and matrix CSV to the
# batch data folder.
#
# Prerequisites: run the batch comparison cell above so that df_pairs exists.
# =============================================================================

import re


def short_name(fname: str) -> str:
    """Derive a concise specimen label from a batch NPZ filename."""
    base = os.path.basename(str(fname))
    base = re.sub(r"_mycocode_bits\.npz$", "", base)
    base = re.sub(r"\.npz$", "", base)
    return base

def make_distance_matrix(df: pd.DataFrame, value_col: str = "hamming_min") -> (pd.DataFrame, np.ndarray, list):
    """Build a symmetric NxN distance matrix from df_pairs."""
    df_ok = df.dropna(subset=[value_col]).copy()
    df_ok["s1"] = df_ok["sample1"].map(short_name)
    df_ok["s2"] = df_ok["sample2"].map(short_name)

    labels = sorted(set(df_ok["s1"]).union(df_ok["s2"]))
    n = len(labels)
    idx = {lab: i for i, lab in enumerate(labels)}
    M = np.full((n, n), np.nan, dtype=float)

    for _, row in df_ok.iterrows():
        i, j = idx[row["s1"]], idx[row["s2"]]
        v = float(row[value_col])
        M[i, j] = v
        M[j, i] = v

    np.fill_diagonal(M, 0.0)
    mat_df = pd.DataFrame(M, index=labels, columns=labels)
    return mat_df, M, labels


if "df_pairs" not in globals():
    raise RuntimeError("df_pairs not found. Run the batch section first to create df_pairs.")

mat_df, M, labels = make_distance_matrix(df_pairs, value_col="hamming_min")

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(max(6, 0.35*len(labels)), max(5, 0.35*len(labels))))
im = ax.imshow(M, aspect="equal", vmin=0, vmax=1)

n = len(labels)
if n <= 40:
    xticks = np.arange(n)
    yticks = np.arange(n)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xticklabels(labels, rotation=90)
    ax.set_yticklabels(labels)
else:
    step = max(1, n // 40)
    xticks = np.arange(0, n, step)
    yticks = np.arange(0, n, step)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xticklabels([labels[i] for i in xticks], rotation=90)
    ax.set_yticklabels([labels[i] for i in yticks])

ax.set_title("Rotation-robust Hamming Distance (hamming_min)")
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Hamming distance (0\u20131)")

plt.tight_layout()
plt.show()

# --- Save heatmap and matrix CSV ---
_out_dir = MYCOCODE_FOLDER if ("MYCOCODE_FOLDER" in globals() and isinstance(MYCOCODE_FOLDER, str) and os.path.isdir(MYCOCODE_FOLDER)) else os.getcwd()
fig_path = os.path.join(_out_dir, "mycocode_hamming_heatmap.png")
csv_path = os.path.join(_out_dir, "mycocode_hamming_matrix.csv")
try:
    fig.savefig(fig_path, dpi=200, bbox_inches="tight")
    mat_df.to_csv(csv_path)
    print(f"Saved heatmap: {fig_path}")
    print(f"Saved matrix CSV: {csv_path}")
except Exception as _e:
    print(f"(Note) Could not save outputs: {_e}")


In [ ]:
# =============================================================================
# Non-Self Nearest-Neighbor Map
# =============================================================================
#
# For each sample, identifies the single closest non-self neighbor by
# hamming_min and renders a binary indicator heatmap with optional
# distance annotations.
#
# Prerequisites: df_pairs must exist from the batch comparison cell.
# =============================================================================

from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# --- Configuration ---
ANNOTATE_DISTANCES = True
FIG_SCALE_PER_NODE = 0.45
EXPORT_PNG = True
EXPORT_FILENAME = "nonself_nearest_neighbor_map.png"

# --- Build label index and symmetric Hamming matrix ---
labels = sorted(set(df_pairs['sample1']).union(set(df_pairs['sample2'])))
idx = {name: k for k, name in enumerate(labels)}
n = len(labels)

H = np.full((n, n), np.nan, dtype=float)
for _, row in df_pairs.dropna(subset=['hamming_min']).iterrows():
    i = idx[row['sample1']]
    j = idx[row['sample2']]
    d = float(row['hamming_min'])
    H[i, j] = d
    H[j, i] = d

# Exclude self-comparisons
H_work = H.copy()
np.fill_diagonal(H_work, np.inf)

# --- Identify nearest non-self neighbor per sample ---
NN_nonself = np.zeros((n, n), dtype=bool)
chosen_dist = np.full((n, n), np.nan, dtype=float)

rows_nn = []
for i, name in enumerate(labels):
    row = H_work[i, :]
    finite_idx = np.where(np.isfinite(row))[0]
    if finite_idx.size == 0:
        rows_nn.append({
            "sample": name,
            "nearest_nonself": None,
            "hamming_min_to_nearest_nonself": np.nan
        })
        continue

    j_star = int(finite_idx[np.argmin(row[finite_idx])])
    d_star = H[i, j_star]

    NN_nonself[i, j_star] = True
    chosen_dist[i, j_star] = d_star

    rows_nn.append({
        "sample": name,
        "nearest_nonself": labels[j_star],
        "hamming_min_to_nearest_nonself": d_star
    })

df_nn_nonself = pd.DataFrame(rows_nn)
print("\nNon-self nearest neighbor per sample:")
display(df_nn_nonself)

# --- Plot ---
fig, ax = plt.subplots(figsize=(max(6, n * FIG_SCALE_PER_NODE),
                                max(5, n * FIG_SCALE_PER_NODE)))

cmap = ListedColormap(['white', 'tab:blue'])
im = ax.imshow(NN_nonself.astype(int), cmap=cmap, vmin=0, vmax=1,
               aspect='equal', interpolation='nearest')
ax.set_title("Non-Self Nearest Neighbor (row i \u2192 column j*)")
ax.set_xticks(range(n))
ax.set_xticklabels(labels, rotation=90)
ax.set_yticks(range(n))
ax.set_yticklabels(labels)

handles = [
    mpatches.Patch(color='tab:blue', label='Nearest neighbor (non-self)'),
    mpatches.Patch(color='white',    label='Not selected')
]
ax.legend(handles=handles, loc='upper right', frameon=False)

if ANNOTATE_DISTANCES:
    for i in range(n):
        j_idxs = np.where(NN_nonself[i])[0]
        if j_idxs.size:
            j = int(j_idxs[0])
            val = chosen_dist[i, j]
            if np.isfinite(val):
                ax.text(j, i, f"{val:.3f}", ha='center', va='center', fontsize=8, color='black')

plt.tight_layout()

if EXPORT_PNG:
    save_dir = MYCOCODE_FOLDER if 'MYCOCODE_FOLDER' in globals() and os.path.isdir(MYCOCODE_FOLDER) else os.getcwd()
    save_path = os.path.join(save_dir, EXPORT_FILENAME)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Plot exported to: {save_path}")

plt.show()


In [ ]:
# =============================================================================
# Pupil-Boundary Distance (Rotation-Invariant) + Score Fusion + Metrics
# =============================================================================
#
# Extracts a rotation-invariant descriptor from the inner (pupil) boundary
# of the unwrapped mask, computes pairwise distances, and fuses them with
# Hamming distances to produce a combined score.
#
# Quantitative discrimination metrics (AUC and d-prime) and optional ROC
# curves are computed for Hamming, pupil, and fused scores.
#
# Prerequisites:
#   - MYCOCODE_FOLDER with *_mycocode_bits.npz and aligned mask PNGs
#   - df_pairs from the batch Hamming comparison
# =============================================================================

import re
import glob

# Optional smoothing
try:
    from scipy.signal import savgol_filter
    _HAS_SG = True
except Exception:
    _HAS_SG = False

# Optional sklearn for ROC/AUC
try:
    from sklearn.metrics import roc_auc_score, roc_curve
    _HAS_SK = True
except Exception:
    _HAS_SK = False

# --- Configuration ---
MASK_GLOB         = "*_unwrapped_mask_aligned.png"
SMOOTH_WINDOW     = 9       # Savitzky-Golay window (odd); 0 to disable
SMOOTH_POLY       = 2
FOURIER_KEEP      = 64      # Lowest non-DC Fourier magnitudes to retain
DIST_METRIC       = "cosine"
BLEND_ALPHA       = 0.7     # fused = alpha * hamming + (1-alpha) * pupil
MIN_VALID_FRAC    = 0.65    # Minimum valid-column fraction for pupil trace

EXPORT_METRICS_CSV = True
EXPORT_ROC_PNG     = True


def short_name(fname: str) -> str:
    """Derive a concise specimen label from a batch NPZ filename."""
    base = os.path.basename(str(fname))
    base = re.sub(r"_mycocode_bits\.npz$", "", base)
    base = re.sub(r"\.npz$", "", base)
    return base

def mask_path_for(sample_label: str, folder: str) -> str:
    """Map a sample label to its aligned unwrapped mask PNG."""
    candidates = glob.glob(os.path.join(folder, MASK_GLOB))
    exact = [p for p in candidates if os.path.basename(p).startswith(sample_label)]
    if len(exact) == 1:
        return exact[0]
    if len(exact) > 1:
        exact.sort(key=lambda p: len(os.path.commonprefix([os.path.basename(p), sample_label])), reverse=True)
        return exact[0]
    sub = [p for p in candidates if sample_label in os.path.basename(p)]
    return sub[0] if sub else None

def load_mask(path: str) -> np.ndarray:
    """Load a binary mask PNG as a uint8 array (rows=radius, cols=angle)."""
    import imageio.v2 as iio
    img = iio.imread(path)
    if img.ndim == 3:
        img = img[..., 0]
    return (img > 0).astype(np.uint8)

def radial_pupil_boundary(mask_unwrapped: np.ndarray):
    """Find the first valid-mask row (inner boundary) for each angle column."""
    R, A = mask_unwrapped.shape
    r_idx = np.full(A, np.nan, dtype=float)
    for a in range(A):
        col = mask_unwrapped[:, a]
        hits = np.flatnonzero(col == 1)
        if hits.size:
            r_idx[a] = float(hits[0])
    valid = np.isfinite(r_idx)
    return r_idx, valid

def preprocess_trace(r_idx: np.ndarray, valid: np.ndarray) -> np.ndarray:
    """Gap-fill, smooth, and z-normalize the pupil boundary trace."""
    x = r_idx.copy()
    if np.any(~valid):
        a = np.arange(x.size)
        if np.any(valid):
            x[~valid] = np.interp(a[~valid], a[valid], x[valid], left=np.nan, right=np.nan)
    if SMOOTH_WINDOW and _HAS_SG:
        finite = np.isfinite(x)
        if finite.sum() >= max(SMOOTH_WINDOW, 10):
            fill = np.interp(np.arange(x.size), np.where(finite)[0], x[finite])
            x = savgol_filter(fill, SMOOTH_WINDOW, SMOOTH_POLY, mode='wrap')
    finite = np.isfinite(x)
    if finite.any():
        mu = np.nanmean(x[finite])
        sd = np.nanstd(x[finite]) if np.nanstd(x[finite]) > 1e-8 else 1.0
        x[finite] = (x[finite] - mu) / sd
    return x

def fourier_magnitude(trace: np.ndarray, keep: int) -> np.ndarray:
    """Rotation-invariant descriptor: magnitude spectrum of rFFT (DC excluded)."""
    t = trace.copy()
    if np.any(~np.isfinite(t)):
        idx = np.arange(t.size)
        finite = np.isfinite(t)
        if finite.sum() >= 2:
            t[~finite] = np.interp(idx[~finite], idx[finite], t[finite])
        else:
            return np.zeros(keep, dtype=float)
    F = np.fft.rfft(t)
    mag = np.abs(F)[1:keep+1]
    if mag.size < keep:
        mag = np.pad(mag, (0, keep - mag.size), mode='constant')
    norm = np.linalg.norm(mag)
    if norm > 0:
        mag = mag / norm
    return mag.astype(np.float32)

def pairwise_distance(X: np.ndarray, metric: str = "cosine") -> np.ndarray:
    """Compute an NxN symmetric distance matrix from feature rows."""
    from numpy.linalg import norm
    N, D = X.shape
    if metric == "cosine":
        Z = X / np.maximum(norm(X, axis=1, keepdims=True), 1e-12)
        S = Z @ Z.T
        M = 1.0 - np.clip(S, -1, 1)
    else:
        G = X @ X.T
        d2 = np.diag(G)[:, None] + np.diag(G)[None, :] - 2 * G
        d2 = np.maximum(d2, 0.0)
        M = np.sqrt(d2)
        M = M / (np.nanmax(M) + 1e-12)
    np.fill_diagonal(M, 0.0)
    return M

def d_prime(same: np.ndarray, diff: np.ndarray) -> float:
    """Signal-detection sensitivity index d-prime."""
    same = np.asarray(same); diff = np.asarray(diff)
    mu1, mu2 = float(np.mean(same)), float(np.mean(diff))
    s1, s2 = float(np.var(same, ddof=1)), float(np.var(diff, ddof=1))
    denom = np.sqrt(max(1e-12, 0.5*(s1 + s2)))
    return abs(mu1 - mu2) / denom

def auc_mann_whitney(y_true: np.ndarray, scores: np.ndarray) -> float:
    """ROC-AUC via Mann-Whitney U (fallback when sklearn is unavailable)."""
    y_true = np.asarray(y_true).astype(bool)
    pos = scores[y_true]
    neg = scores[~y_true]
    if pos.size == 0 or neg.size == 0:
        return np.nan
    ranks = scores.argsort().argsort() + 1
    r_pos = ranks[y_true]
    n_pos, n_neg = pos.size, neg.size
    U = r_pos.sum() - n_pos*(n_pos+1)/2
    auc = U / (n_pos * n_neg)
    return float(auc)


# --- Build sample list ---
if "df_pairs" not in globals():
    raise RuntimeError("df_pairs not found. Run the batch section first.")

samples = sorted(set(df_pairs["sample1"]).union(set(df_pairs["sample2"])))
labels = [short_name(s) for s in samples]
mask_paths = {lab: mask_path_for(lab, MYCOCODE_FOLDER) for lab in labels}

# --- Extract pupil-boundary descriptors ---
rows_feat, features, valid_fracs = [], [], []
angles_count = None

for lab in labels:
    p = mask_paths.get(lab)
    if not p or (not os.path.isfile(p)):
        rows_feat.append({"sample": lab, "mask_path": p, "valid_frac": 0.0, "notes": "mask not found"})
        features.append(np.zeros(FOURIER_KEEP, dtype=float))
        valid_fracs.append(0.0)
        continue

    m = load_mask(p)
    if angles_count is None:
        angles_count = m.shape[1]
    r_idx, valid_cols = radial_pupil_boundary(m)
    trace = preprocess_trace(r_idx, valid_cols)
    valid_frac = float(np.mean(np.isfinite(r_idx)))
    mag = fourier_magnitude(trace, FOURIER_KEEP)

    rows_feat.append({"sample": lab, "mask_path": p, "valid_frac": valid_frac, "notes": ""})
    features.append(mag)
    valid_fracs.append(valid_frac)

df_feat = pd.DataFrame(rows_feat)
X = np.vstack(features)

low_valid = df_feat["valid_frac"].values < MIN_VALID_FRAC
if low_valid.any():
    print(f"Note: {low_valid.sum()} sample(s) have low pupil validity (<{MIN_VALID_FRAC:.2f}). Distances may be noisy.")

# --- Pairwise pupil distance ---
M_pupil = pairwise_distance(X, metric=DIST_METRIC)

out_dir = MYCOCODE_FOLDER if os.path.isdir(MYCOCODE_FOLDER) else os.getcwd()
df_feat.to_csv(os.path.join(out_dir, "df_pupil_features.csv"), index=False)
pd.DataFrame(M_pupil, index=labels, columns=labels).to_csv(
    os.path.join(out_dir, "mycocode_pupil_dist_matrix.csv"))

# --- Fuse Hamming and pupil scores ---
def get_idx(name):
    return labels.index(short_name(name))

pupil_dists = []
for _, row in df_pairs.iterrows():
    if pd.isna(row.get("hamming_min")):
        pupil_dists.append(np.nan)
    else:
        i1 = get_idx(row["sample1"]); i2 = get_idx(row["sample2"])
        pupil_dists.append(M_pupil[i1, i2])

df_pairs["pupil_dist_raw"] = pupil_dists

# Rescale pupil distance to [0, 1] for blending
pupil_vals = df_pairs["pupil_dist_raw"].values
finite = np.isfinite(pupil_vals)
if finite.any():
    lo, hi = np.nanpercentile(pupil_vals[finite], [5, 95])
    span = max(hi - lo, 1e-6)
    df_pairs["pupil_dist01"] = np.clip((df_pairs["pupil_dist_raw"] - lo) / span, 0, 1)
else:
    df_pairs["pupil_dist01"] = np.nan

df_pairs["score_fused"] = BLEND_ALPHA * df_pairs["hamming_min"] + (1 - BLEND_ALPHA) * df_pairs["pupil_dist01"]

lab_to_valid = dict(zip(df_feat["sample"], df_feat["valid_frac"]))
df_pairs["pupil_valid_pair_min"] = df_pairs.apply(
    lambda r: min(lab_to_valid.get(short_name(r["sample1"]), 0.0),
                  lab_to_valid.get(short_name(r["sample2"]), 0.0)), axis=1)

df_pairs["is_self"] = df_pairs["sample1"] == df_pairs["sample2"]
df_pairs.to_csv(os.path.join(out_dir, "df_pairs_fused.csv"), index=False)

# --- Diagnostic plots ---
plt.figure(figsize=(5,4))
sel = df_pairs["pupil_valid_pair_min"] >= MIN_VALID_FRAC
plt.scatter(df_pairs.loc[sel, "hamming_min"], df_pairs.loc[sel, "pupil_dist01"], s=20, alpha=0.7)
plt.xlabel("Hamming_min (rotation-robust)")
plt.ylabel("Pupil distance (0\u20131, rescaled)")
plt.title("Orthogonality check: texture vs. pupil boundary")
plt.tight_layout()
plt.show()

# Fused distance heatmap
labs_sorted = sorted(set(short_name(s) for s in df_pairs["sample1"]).union(set(short_name(s) for s in df_pairs["sample2"])))
L = len(labs_sorted)
F = np.full((L, L), np.nan, dtype=float)
idxL = {lab:i for i, lab in enumerate(labs_sorted)}
for _, r in df_pairs.dropna(subset=["score_fused"]).iterrows():
    i = idxL[short_name(r["sample1"])]; j = idxL[short_name(r["sample2"])]
    F[i,j] = r["score_fused"]; F[j,i] = r["score_fused"]
np.fill_diagonal(F, 0.0)

plt.figure(figsize=(max(6, 0.35*L), max(5, 0.35*L)))
im = plt.imshow(F, vmin=0, vmax=1, aspect="equal")
plt.xticks(range(L), labs_sorted, rotation=90)
plt.yticks(range(L), labs_sorted)
plt.title(f"Fused distance heatmap (\u03b1={BLEND_ALPHA:.2f})")
cbar = plt.colorbar(im, fraction=0.046, pad=0.04); cbar.set_label("Fused score (0\u20131)")
plt.tight_layout(); plt.show()

# --- Discrimination metrics ---
mask_eval = (df_pairs["pupil_valid_pair_min"] >= MIN_VALID_FRAC) & \
            df_pairs[["hamming_min","pupil_dist01","score_fused"]].notna().all(axis=1)

sub = df_pairs.loc[mask_eval, ["is_self","hamming_min","pupil_dist01","score_fused"]].copy()
if sub.empty:
    print("No valid pairs for metrics (check masks/validity thresholds).")
else:
    y = sub["is_self"].astype(int).values
    s_h = (-sub["hamming_min"].values)
    s_p = (-sub["pupil_dist01"].values)
    s_f = (-sub["score_fused"].values)

    if _HAS_SK:
        auc_h = roc_auc_score(y, s_h)
        auc_p = roc_auc_score(y, s_p)
        auc_f = roc_auc_score(y, s_f)
    else:
        auc_h = auc_mann_whitney(y, s_h)
        auc_p = auc_mann_whitney(y, s_p)
        auc_f = auc_mann_whitney(y, s_f)

    d_h = d_prime(sub.loc[sub.is_self,"hamming_min"].values,
                  sub.loc[~sub.is_self,"hamming_min"].values)
    d_f = d_prime(sub.loc[sub.is_self,"score_fused"].values,
                  sub.loc[~sub.is_self,"score_fused"].values)

    print(f"AUC (Hamming): {auc_h:.3f} | AUC (Pupil): {auc_p:.3f} | AUC (Fused): {auc_f:.3f}")
    print(f"d'  (Hamming): {d_h:.2f}  | d'  (Fused): {d_f:.2f}  | Lift: {((d_f-d_h)/max(1e-9,d_h))*100:.1f}%")

    if EXPORT_METRICS_CSV:
        metrics_df = pd.DataFrame({
            "metric": ["AUC_hamming","AUC_pupil","AUC_fused","dprime_hamming","dprime_fused","dprime_lift_pct"],
            "value":  [auc_h, auc_p, auc_f, d_h, d_f, ((d_f-d_h)/max(1e-9,d_h))*100.0]
        })
        metrics_df.to_csv(os.path.join(out_dir, "metrics_summary.csv"), index=False)

    if _HAS_SK:
        plt.figure(figsize=(5,5))
        for label,score in [
            ("Hamming", s_h),
            ("Pupil",   s_p),
            ("Fused",   s_f),
        ]:
            fpr, tpr, _ = roc_curve(y, score)
            plt.plot(fpr, tpr, label=label)
        plt.plot([0,1],[0,1], "--", lw=0.8, color="k")
        plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
        plt.title("ROC curves: Hamming vs Pupil vs Fused")
        plt.legend()
        plt.tight_layout()
        if EXPORT_ROC_PNG:
            plt.savefig(os.path.join(out_dir, "roc_curves.png"), dpi=300, bbox_inches="tight")
        plt.show()
    else:
        print("(Note) sklearn not available; AUC computed via Mann-Whitney, ROC curves skipped.")

print("Saved:",
      os.path.join(out_dir, "df_pupil_features.csv"),
      os.path.join(out_dir, "mycocode_pupil_dist_matrix.csv"),
      os.path.join(out_dir, "df_pairs_fused.csv"),
      os.path.join(out_dir, "metrics_summary.csv") if EXPORT_METRICS_CSV else "(metrics CSV disabled)")
